# LLM101: Building a Large Language Model From Scratch

**A ~15M-parameter GPT-style decoder-only Transformer in ~800 lines of PyTorch.**

This notebook walks through every stage of building, training, and using a language model from scratch. Each component is written explicitly -- RMSNorm, Rotary Position Embeddings (RoPE), SwiGLU FFN, causal self-attention with a combined QKV projection, and weight tying -- so you can see exactly how modern LLMs work inside.

The architecture mirrors modern production models (LLaMA / Mistral / Qwen), scaled down to a size that trains on a single GPU in minutes.

**Aligned with:** Sebastian Raschka's *Build a Large Language Model (From Scratch)*

---

### What you will learn

| Section | Topic | Key Concept |
|---------|-------|-------------|
| 0 | Setup & GPU Detection | Environment provisioning |
| 1 | Configuration | All hyperparameters in one place |
| 2 | BPE Tokenizer | Byte-level Byte Pair Encoding from scratch |
| 3 | Dataset | Sliding-window causal LM data pipeline |
| 4 | Model Architecture | RMSNorm, RoPE, Attention, SwiGLU, TransformerBlock, NanoLLM |
| 5 | Forward Pass Walkthrough | Hook-based intermediate capture and visualization |
| 6 | Training | AdamW, warmup + cosine decay, mixed precision |
| 7 | Text Generation | Autoregressive sampling with temperature, top-k, top-p |
| 8 | Attention Visualization | All-heads grid across layers |
| 9 | KV Cache Deep Dive | Prefill + decode pattern, equivalence proof, speedup |

## Section 0: Setup & GPU Detection

The first thing any deep learning project needs is to know what hardware it is running on. Our model is designed for CUDA GPUs, but can fall back to CPU for demonstration purposes (training will be much slower).

We set `NANOLLM_ALLOW_CPU=1` if no GPU is detected so the notebook runs everywhere.

In [ ]:
import os, sys, subprocess

# ── Google Colab / remote environment setup ──
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    REPO_DIR = "/content/llm101"
    if not os.path.exists(REPO_DIR):
        print("Cloning LLM101 repository...")
        result = subprocess.run(
            ["git", "clone", "https://github.com/rahulbasu-dev/llm101.git", REPO_DIR],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"Clone failed: {result.stderr}")
            raise RuntimeError("Could not clone repo. Is it public? Check the URL.")
        print("Clone successful.")
    os.chdir(REPO_DIR)
    os.system("pip install -q matplotlib numpy tqdm")
    PROJECT_ROOT = REPO_DIR
else:
    PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch

if not torch.cuda.is_available():
    os.environ["NANOLLM_ALLOW_CPU"] = "1"
    print("⚠ No CUDA GPU detected — running on CPU.")
    print("  Training will be slow. For full speed, use Colab with a GPU runtime.\n")
    if IN_COLAB:
        print("  Tip: Runtime → Change runtime type → GPU (T4 is free)\n")
else:
    props = torch.cuda.get_device_properties(0)
    print(f"✓ GPU detected: {props.name}")
    print(f"  VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"  CUDA: {torch.version.cuda}")
    bf16 = "supported" if torch.cuda.is_bf16_supported() else "not supported"
    print(f"  BF16: {bf16}\n")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
from IPython.display import display
# Core imports — these are ALL from the LLM101 project, not copy-pasted
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import math
import time
import torch.nn as nn
import torch.nn.functional as F

from config import NanoLLMConfig, safe_savefig
from tokenizer import BPETokenizer, NUM_BASE, NUM_SPECIAL, BYTE_OFFSET, PAD_ID, BOS_ID, EOS_ID, UNK_ID
from dataset import TextDataset, create_dataloader
from model import NanoLLM, RMSNorm, RotaryPositionEmbedding, CausalSelfAttention, FeedForward, TransformerBlock, _sample_from_logits
from teach import ForwardCapture, token_labels

print(f"All modules imported successfully from {os.getcwd()}")

## Section 1: Configuration

Every hyperparameter lives in one `NanoLLMConfig` dataclass. This single source of truth is imported by every module -- tokenizer, dataset, model, training loop, and generation.

**Key design choices:**
- **d_model=384, n_heads=6** -- gives d_head=64, matching the standard attention head size used in GPT-2 through LLaMA-3. Small enough to train in minutes; large enough to learn real patterns.
- **n_layers=6** -- enough depth for the model to develop distinct layer behaviors (early layers learn local patterns, later layers learn composition).
- **max_seq_len=256** -- context window. Also sizes the RoPE cos/sin tables and the causal mask buffer.
- **target_vocab_size=4096** -- BPE merge target. Small corpus = smaller vocab. The base is 260 (4 special + 256 byte tokens).
- **Weight tying** -- the embedding matrix and lm_head share the same weights, reducing parameters and improving quality (Press & Wolf, 2017).

In [ ]:
config = NanoLLMConfig()

# Display all hyperparameters in a formatted table
print("=" * 60)
print("  NanoLLM Configuration")
print("=" * 60)

sections = {
    "Model Architecture": [
        ("d_model (hidden dim)", config.d_model),
        ("n_layers", config.n_layers),
        ("n_heads", config.n_heads),
        ("d_head (derived)", config.d_head),
        ("d_ff (FFN intermediate)", config.d_ff),
        ("max_seq_len", config.max_seq_len),
        ("dropout", config.dropout),
    ],
    "Tokenizer": [
        ("target_vocab_size", config.target_vocab_size),
        ("base tokens (special+bytes)", NUM_BASE),
        ("merges needed", config.target_vocab_size - NUM_BASE),
    ],
    "Training": [
        ("batch_size", config.batch_size),
        ("learning_rate", config.learning_rate),
        ("weight_decay", config.weight_decay),
        ("max_epochs", config.max_epochs),
        ("warmup_steps", config.warmup_steps),
        ("grad_clip", config.grad_clip),
    ],
    "Generation": [
        ("temperature", config.temperature),
        ("top_k", config.top_k),
        ("top_p", config.top_p),
    ],
}

for section, params in sections.items():
    print(f"\n  {section}")
    print("  " + "-" * 40)
    for name, val in params:
        print(f"    {name:<30} {val}")

print(f"\n  Device: {config.device}")
print(f"  AMP dtype: {config.amp_dtype}")
print("=" * 60)

## Section 2: Byte-Level BPE Tokenizer

Before a language model can process text, we need to convert characters into numbers. Our tokenizer uses **Byte Pair Encoding (BPE)** -- the same algorithm used by GPT-2, GPT-3, and LLaMA.

**How BPE works:**
1. Start with individual bytes (256 possible values) as the base vocabulary
2. Count all adjacent pairs in the corpus
3. Merge the most frequent pair into a new token
4. Repeat until we reach our target vocabulary size

**Vocabulary layout:**
- Tokens 0-3: Special tokens (`<PAD>`, `<BOS>`, `<EOS>`, `<UNK>`)
- Tokens 4-259: Raw bytes (0x00 through 0xFF)
- Tokens 260+: Learned BPE merge tokens

This means our tokenizer can handle ANY byte sequence -- no "unknown character" problems.

In [ ]:
# Download TinyStories corpus if not already present
CORPUS_PATH = config.data_path

if not os.path.exists(CORPUS_PATH):
    print("Downloading TinyStories dataset (Microsoft Research)...")
    os.makedirs("data", exist_ok=True)
    import urllib.request
    url = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt"
    raw_path = CORPUS_PATH + ".raw"
    urllib.request.urlretrieve(url, raw_path)
    # Truncate to ~1.5MB for fast training (full file is ~27MB)
    with open(raw_path, "r", encoding="utf-8") as rf:
        text = rf.read(1_500_000)
    with open(CORPUS_PATH, "w", encoding="utf-8") as wf:
        wf.write(text)
    os.remove(raw_path)
    print(f"Downloaded to {CORPUS_PATH}")

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Corpus: {len(raw_text):,} characters")
print(f"First 200 characters:\n{raw_text[:200]}")

In [ ]:
# Train the BPE tokenizer (or load if already saved)
tokenizer = BPETokenizer(target_vocab_size=config.target_vocab_size)

if os.path.exists(config.tokenizer_path):
    tokenizer.load(config.tokenizer_path)
    print("Loaded pre-trained tokenizer.")
else:
    print("Training BPE tokenizer from scratch (this takes ~30s)...")
    actual_vocab = tokenizer.train(raw_text)
    tokenizer.save(config.tokenizer_path)

config.vocab_size = tokenizer.vocab_size
print(f"\nFinal vocab size: {config.vocab_size}")
print(f"  = {NUM_SPECIAL} special + 256 bytes + {len(tokenizer.merges)} merges")

In [ ]:
# Demonstrate encode/decode round-trip
test_text = "To be, or not to be, that is the question."
encoded = tokenizer.encode(test_text, add_special=False)
decoded = tokenizer.decode(encoded)

print(f"Original:  {test_text!r}")
print(f"Token IDs: {encoded}")
print(f"Decoded:   {decoded!r}")
print(f"Round-trip match: {test_text == decoded}")
print(f"\nToken breakdown:")
for tid in encoded:
    label = tokenizer.decode_token(tid)
    is_merge = tid >= NUM_BASE
    kind = "MERGE" if is_merge else ("BYTE" if tid >= BYTE_OFFSET else "SPECIAL")
    print(f"  {tid:>5}  {kind:<7}  {label!r}")

In [ ]:
# Visualize vocabulary layout
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: vocabulary composition bar chart
categories = ["Special\n(PAD,BOS,EOS,UNK)", "Byte tokens\n(0x00-0xFF)", "BPE merges\n(learned)"]
counts = [NUM_SPECIAL, 256, len(tokenizer.merges)]
colors = ["#dc3545", "#fd8d3c", "#28a745"]
axes[0].bar(categories, counts, color=colors, edgecolor="white", linewidth=2)
for i, (c, v) in enumerate(zip(categories, counts)):
    axes[0].text(i, v + 30, str(v), ha="center", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of tokens")
axes[0].set_title("Vocabulary Composition", fontsize=13, fontweight="bold")
axes[0].grid(True, axis="y", alpha=0.3)

# Right: compression ratio on sample texts
samples = [
    "Hello world!",
    "The king said to his servant",
    "ROMEO:\nBut, soft! what light through yonder window breaks?",
    "abcdefghijklmnop",
]
ratios = []
for s in samples:
    toks = tokenizer.encode(s, add_special=False)
    ratios.append(len(s.encode("utf-8")) / len(toks))

y_pos = range(len(samples))
bars = axes[1].barh(y_pos, ratios, color="#08519c")
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels([s[:35] + ("..." if len(s) > 35 else "") for s in samples], fontsize=9)
axes[1].set_xlabel("Compression ratio (bytes / tokens)")
axes[1].set_title("BPE Compression Ratio", fontsize=13, fontweight="bold")
for i, v in enumerate(ratios):
    axes[1].text(v + 0.05, i, f"{v:.2f}x", va="center", fontsize=10)
axes[1].grid(True, axis="x", alpha=0.3)

fig.tight_layout()
display(fig)
plt.close(fig)

## Section 3: Dataset -- Sliding Windows

For causal language modeling, we create (input, target) pairs where the target is the input shifted right by one position. The model learns to predict the next token at every position simultaneously.

**Sliding window approach:**
- We slide a window of length `seq_len` across the tokenized corpus
- Default stride = `seq_len // 2` (50% overlap) for better data utilization
- Each window produces one training example

```
tokens:  [A] [B] [C] [D] [E] [F] [G] [H] ...
          |----window 1----|
                  |----window 2----|

window 1: input  = [A, B, C, D]    target = [B, C, D, E]
window 2: input  = [C, D, E, F]    target = [D, E, F, G]
```

At every position, the model predicts the NEXT token -- this is autoregressive training.

In [ ]:
# Tokenize the full corpus (no special tokens -- we add BOS/EOS per-sequence later if needed)
all_tokens = tokenizer.encode(raw_text, add_special=False)
print(f"Corpus: {len(raw_text):,} chars -> {len(all_tokens):,} tokens")
print(f"Compression ratio: {len(raw_text) / len(all_tokens):.2f}x")

# 90/10 train/val split (sequential, not random -- preserves text structure)
split_idx = int(0.9 * len(all_tokens))
train_tokens = all_tokens[:split_idx]
val_tokens = all_tokens[split_idx:]
print(f"\nTrain: {len(train_tokens):,} tokens")
print(f"Val:   {len(val_tokens):,} tokens")

# Create datasets
train_dataset = TextDataset(train_tokens, config.max_seq_len)
val_dataset = TextDataset(val_tokens, config.max_seq_len)

In [ ]:
# Visualize the input/target shift and overlapping windows
inp, tgt = train_dataset[0]

fig, axes = plt.subplots(2, 1, figsize=(14, 5))

# Top: show the input->target shift for one sample
show_n = 12  # show first N positions for clarity
positions = list(range(show_n))
inp_labels = [tokenizer.decode_token(int(t)).replace("\n", "\\n") for t in inp[:show_n]]
tgt_labels = [tokenizer.decode_token(int(t)).replace("\n", "\\n") for t in tgt[:show_n]]

axes[0].set_xlim(-0.5, show_n - 0.5)
axes[0].set_ylim(-0.5, 1.5)
for i in range(show_n):
    # Input row
    axes[0].text(i, 1, f"{inp_labels[i]}", ha="center", va="center", fontsize=9,
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="#e6f2ff", edgecolor="#4a90d9"))
    # Target row
    axes[0].text(i, 0, f"{tgt_labels[i]}", ha="center", va="center", fontsize=9,
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="#ffe6e6", edgecolor="#dc3545"))
    # Arrow
    axes[0].annotate("", xy=(i, 0.35), xytext=(i, 0.65),
                     arrowprops=dict(arrowstyle="->", color="#333", lw=1.5))

axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(["Target\n(shifted +1)", "Input"], fontsize=10)
axes[0].set_xticks(positions)
axes[0].set_xticklabels([f"pos {i}" for i in positions], fontsize=8)
axes[0].set_title("Input vs Target: target = input shifted right by 1", fontsize=12, fontweight="bold")

# Bottom: show overlapping windows
n_windows = 4
stride = config.max_seq_len // 2
colors_w = ["#08519c", "#e6550d", "#2ca02c", "#9467bd"]
for w in range(n_windows):
    start = w * stride
    axes[1].barh(w, config.max_seq_len, left=start, height=0.6,
                 color=colors_w[w % len(colors_w)], alpha=0.7, edgecolor="white")
    axes[1].text(start + config.max_seq_len / 2, w,
                 f"Window {w} (tokens {start}-{start + config.max_seq_len - 1})",
                 ha="center", va="center", fontsize=9, color="white", fontweight="bold")
axes[1].set_xlabel("Token position in corpus")
axes[1].set_yticks(range(n_windows))
axes[1].set_yticklabels([f"Sample {i}" for i in range(n_windows)])
axes[1].set_title(f"Overlapping sliding windows (stride={stride}, seq_len={config.max_seq_len})",
                  fontsize=12, fontweight="bold")

fig.tight_layout()
display(fig)
plt.close(fig)

## Section 4: Model Architecture (Bottom-Up)

We build the model from its smallest components up to the full Transformer. Each component is implemented explicitly in `model.py` -- no hidden PyTorch built-ins.

```
NanoLLM
  |-- token_emb (nn.Embedding)
  |-- emb_dropout
  |-- blocks (x6 TransformerBlocks)
  |     |-- attn_norm (RMSNorm)
  |     |-- attn (CausalSelfAttention)
  |     |     |-- qkv_proj, out_proj, rope, causal_mask
  |     |-- ffn_norm (RMSNorm)
  |     |-- ffn (FeedForward / SwiGLU)
  |           |-- gate_proj, up_proj, down_proj
  |-- norm_f (RMSNorm)
  |-- lm_head (weight-tied with token_emb)
```

### 4a: RMSNorm

RMSNorm (Root Mean Square Normalization) is simpler and faster than LayerNorm:
- **LayerNorm**: subtract mean, divide by std, apply scale + bias
- **RMSNorm**: divide by root-mean-square, apply scale only (no mean subtraction, no bias)

Used in LLaMA, LLaMA-2, Mistral, Qwen, and Gemma.

In [ ]:
# 4a: RMSNorm -- instantiate and run dummy data
norm = RMSNorm(config.d_model)
dummy = torch.randn(2, 10, config.d_model)  # (batch=2, seq_len=10, d_model=384)
normed = norm(dummy)

print("RMSNorm")
print(f"  Formula: output = x * rsqrt(mean(x^2) + eps) * weight")
print(f"  Input shape:  {tuple(dummy.shape)}")
print(f"  Output shape: {tuple(normed.shape)}")
print(f"  Learnable params: {norm.weight.shape[0]} (scale gamma, one per dimension)")
print(f"\n  Input  - mean: {dummy.mean():.4f}, std: {dummy.std():.4f}")
print(f"  Output - mean: {normed.mean():.4f}, std: {normed.std():.4f}")

# Compare to LayerNorm
ln = nn.LayerNorm(config.d_model)
ln_out = ln(dummy)
print(f"\n  Comparison -- LayerNorm has {sum(p.numel() for p in ln.parameters())} params (weight + bias)")
print(f"  RMSNorm has {sum(p.numel() for p in norm.parameters())} params (weight only)")

### 4b: Rotary Position Embeddings (RoPE)

RoPE encodes absolute position by **rotating** Q and K vectors, but the dot product `Q . K` naturally captures **relative** position. This elegant design is used in LLaMA, Mistral, Qwen, PaLM, and Phi.

For each pair of dimensions `(2i, 2i+1)`, rotate by angle `pos * theta_i` where `theta_i = 10000^(-2i / d_head)`.

Key insight: low-frequency dimensions rotate slowly (capture long-range relationships), high-frequency dimensions rotate quickly (capture local patterns).

In [ ]:
# 4b: RoPE -- show the cos/sin tables and rotation effect
rope = RotaryPositionEmbedding(config.d_head, config.max_seq_len)

# Apply to dummy Q vectors
q_dummy = torch.randn(1, config.n_heads, 16, config.d_head)  # (B, heads, T=16, d_head)
q_rotated = rope(q_dummy, seq_len=16, start_pos=0)

print("RotaryPositionEmbedding")
print(f"  d_head: {config.d_head}  ->  {config.d_head // 2} frequency bands")
print(f"  max_seq_len: {config.max_seq_len}")
print(f"  cos_cached shape: {tuple(rope.cos_cached.shape)}")
print(f"  Input Q shape:    {tuple(q_dummy.shape)}")
print(f"  Rotated Q shape:  {tuple(q_rotated.shape)}")

# Visualize the cos/sin tables
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cos_table = rope.cos_cached.numpy()
sin_table = rope.sin_cached.numpy()
show_pos = min(64, config.max_seq_len)

im1 = axes[0].imshow(cos_table[:show_pos], aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("cos(pos * theta_i)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Frequency index i")
axes[0].set_ylabel("Position")
plt.colorbar(im1, ax=axes[0], shrink=0.8)

im2 = axes[1].imshow(sin_table[:show_pos], aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("sin(pos * theta_i)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Frequency index i")
plt.colorbar(im2, ax=axes[1], shrink=0.8)

# Show rotation of a unit vector at different positions
positions = [0, 2, 4, 8, 16, 32]
positions = [p for p in positions if p < show_pos]
colors_pos = plt.cm.viridis(np.linspace(0.15, 0.85, len(positions)))
for p, color in zip(positions, colors_pos):
    c, s = cos_table[p, 0], sin_table[p, 0]
    axes[2].arrow(0, 0, c, s, head_width=0.04, length_includes_head=True, color=color, linewidth=2)
    axes[2].text(c * 1.15, s * 1.15, f"pos {p}", color=color, fontsize=9, fontweight="bold")
axes[2].set_xlim(-1.3, 1.3)
axes[2].set_ylim(-1.3, 1.3)
axes[2].set_aspect("equal")
axes[2].axhline(0, color="#bbb", linewidth=0.5)
axes[2].axvline(0, color="#bbb", linewidth=0.5)
axes[2].grid(True, alpha=0.3)
axes[2].set_title("Unit vector rotated by position\n(lowest-freq dim pair)", fontsize=12, fontweight="bold")

fig.tight_layout()
display(fig)
plt.close(fig)

### 4c: Causal Self-Attention

The core mechanism of the Transformer. Each token computes a weighted combination of all **preceding** tokens' value vectors, where the weights come from query-key compatibility scores.

**Steps:**
1. Project input `x` into Q, K, V via a single combined linear projection
2. Apply RoPE to Q and K (position encoding)
3. Compute scaled dot-product: `scores = (Q . K^T) / sqrt(d_head)`
4. Apply causal mask (lower triangular) -- token i can only see tokens 0..i
5. Softmax to get attention weights (each row sums to 1)
6. Weighted sum of V vectors
7. Concatenate heads and project output

In [ ]:
# 4c: CausalSelfAttention
attn = CausalSelfAttention(config)
x_dummy = torch.randn(2, 10, config.d_model)  # (B=2, T=10, d_model)
out, kv_cache = attn(x_dummy)

print("CausalSelfAttention")
print(f"  Input:  {tuple(x_dummy.shape)} -- (batch, seq_len, d_model)")
print(f"  Output: {tuple(out.shape)} -- same shape (residual-friendly)")
print(f"  KV cache: k={tuple(kv_cache[0].shape)}, v={tuple(kv_cache[1].shape)}")
print(f"\n  Combined QKV projection: {config.d_model} -> {3 * config.d_model} (single matmul)")
print(f"  n_heads={config.n_heads}, d_head={config.d_head}")
print(f"  Causal mask shape: {tuple(attn.causal_mask.shape)}")

# Show the causal mask
fig, ax = plt.subplots(figsize=(5, 5))
mask_show = attn.causal_mask[0, 0, :12, :12].numpy()
ax.imshow(mask_show, cmap="Blues", vmin=0, vmax=1)
ax.set_title("Causal mask (first 12 positions)\n1 = can attend, 0 = masked", fontsize=12, fontweight="bold")
ax.set_xlabel("Key position (j)")
ax.set_ylabel("Query position (i)")
for i in range(12):
    for j in range(12):
        ax.text(j, i, int(mask_show[i, j]), ha="center", va="center",
                fontsize=9, color="white" if mask_show[i, j] > 0.5 else "black")
fig.tight_layout()
display(fig)
plt.close(fig)

### 4d: SwiGLU Feed-Forward Network

After attention routes information between positions, the FFN computes on it position-wise. This is where the model stores and retrieves factual knowledge.

**SwiGLU** (Shazeer, 2020) replaces the standard `ReLU(x * W1) * W2` with:
```
output = (SiLU(x * W_gate) * (x * W_up)) * W_down
```

SiLU (Sigmoid Linear Unit) = `x * sigmoid(x)`, also called "swish". The gating mechanism gives the model more expressive power. SwiGLU has 3 weight matrices instead of 2, so we use `2/3 * d_ff` for the hidden dimension to keep parameter count comparable.

In [ ]:
# 4d: SwiGLU Feed-Forward
ffn = FeedForward(config)
x_dummy = torch.randn(2, 10, config.d_model)
ffn_out = ffn(x_dummy)

hidden_dim = int(2 * config.d_ff / 3)
hidden_dim = ((hidden_dim + 7) // 8) * 8  # round to multiple of 8

print("FeedForward (SwiGLU)")
print(f"  Input:      {tuple(x_dummy.shape)}")
print(f"  Output:     {tuple(ffn_out.shape)}")
print(f"  d_model:    {config.d_model}")
print(f"  d_ff:       {config.d_ff}")
print(f"  hidden_dim: {hidden_dim} (2/3 * d_ff, rounded to multiple of 8)")
print(f"\n  gate_proj: ({config.d_model}, {hidden_dim}) -- learns WHAT to let through")
print(f"  up_proj:   ({config.d_model}, {hidden_dim}) -- expands the representation")
print(f"  down_proj: ({hidden_dim}, {config.d_model}) -- projects back down")
print(f"  Total FFN params: {sum(p.numel() for p in ffn.parameters()):,}")

### 4e: TransformerBlock -- Pre-Norm Residual

Each block follows the **Pre-Norm** pattern (normalize BEFORE the sublayer, not after):
```
x = x + Attention(RMSNorm(x))    # self-attention with residual
x = x + FFN(RMSNorm(x))          # feed-forward with residual
```

Pre-Norm is more training-stable than Post-Norm. Used in GPT-2+, LLaMA, and all modern LLMs.

### 4f: Full NanoLLM -- Assembling the Model

In [ ]:
# 4e: TransformerBlock
block = TransformerBlock(config)
x_dummy = torch.randn(2, 10, config.d_model)
block_out, block_kv = block(x_dummy)

print("TransformerBlock (Pre-Norm)")
print(f"  Input:  {tuple(x_dummy.shape)}")
print(f"  Output: {tuple(block_out.shape)}")
print(f"  Params: {sum(p.numel() for p in block.parameters()):,}")

# 4f: Full NanoLLM
print("\n" + "=" * 60)
model = NanoLLM(config).to(DEVICE)
model.eval()

# Parameter count with weight-tying awareness
total_params = sum(p.numel() for p in model.parameters())
print(f"\nWeight tying: token_emb.weight is lm_head.weight = {model.lm_head.weight is model.token_emb.weight}")
print(f"  token_emb shape: {tuple(model.token_emb.weight.shape)} = {model.token_emb.weight.numel():,} params")

# Detailed breakdown
print(f"\nParameter breakdown:")
seen_ids = set()
for name, p in model.named_parameters():
    if id(p) not in seen_ids:
        seen_ids.add(id(p))
        print(f"  {name:<45} {str(tuple(p.shape)):<20} {p.numel():>10,}")
print(f"  {'TOTAL':<45} {'':20} {sum(p.numel() for id_p, p in [(id(p), p) for p in model.parameters()] if id_p):>10,}")

In [ ]:
# Quick forward pass sanity check
batch = torch.randint(0, config.vocab_size, (2, 64), device=DEVICE)
targets = torch.randint(0, config.vocab_size, (2, 64), device=DEVICE)

with torch.no_grad():
    logits, loss = model(batch, targets)

print("Forward pass sanity check:")
print(f"  Input:   (2, 64) token IDs")
print(f"  Logits:  {tuple(logits.shape)} -- (batch, seq_len, vocab_size)")
print(f"  Loss:    {loss.item():.4f}")
print(f"  Expected loss at init (random): ~ln({config.vocab_size}) = {math.log(config.vocab_size):.4f}")

## Section 5: Forward Pass Walkthrough

Now we trace a real prompt through the model, capturing intermediate tensors at every stage using `ForwardCapture` hooks from `teach.py`. The hooks do **not** modify the model -- they observe the forward pass via `register_forward_hook` / `register_forward_pre_hook`.

We will visualize:
1. Token embeddings (the raw lookup vectors)
2. Q, K, V projections after RoPE
3. Raw attention scores (before masking)
4. Masked scores (causal mask applied)
5. Softmax attention weights (probabilities)
6. FFN input and output (where knowledge is stored/retrieved)

In [ ]:
# Run a prompt through the model with ForwardCapture hooks
PROMPT = "The cat sat on the"
TARGET_LAYER = 0  # Which layer to inspect in detail
TARGET_HEAD = 0   # Which attention head to focus on

token_ids = tokenizer.encode(PROMPT, add_special=False)
labels = token_labels(tokenizer, token_ids, max_len=8)
idx = torch.tensor([token_ids], device=DEVICE)

print(f"Prompt: {PROMPT!r}")
print(f"Tokens ({len(token_ids)}): {labels}")
print(f"Token IDs: {token_ids}")
print(f"Inspecting: layer={TARGET_LAYER}, head={TARGET_HEAD}")

# Install hooks and run forward pass
capture = ForwardCapture(model, target_layer=TARGET_LAYER)
with torch.no_grad():
    model(idx)
capture.remove()

print(f"\nCaptured tensors: {list(capture.store.keys())}")

In [ ]:
# Visualize: Token Embeddings
emb = capture.store["embeddings"][0].numpy()  # (T, d_model)
T_len, d = emb.shape
show_d = min(48, d)

fig, ax = plt.subplots(figsize=(14, 3 + 0.25 * T_len))
vmax = abs(emb).max()
im = ax.imshow(emb[:, :show_d], aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_yticks(range(T_len))
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel(f"Embedding dimension (first {show_d} of {d})")
ax.set_ylabel("Token")
ax.set_title("Token Embeddings -- each token becomes a 384-dim vector", fontsize=13, fontweight="bold")
plt.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Visualize: Q, K, V projections
q = capture.store["q"][0, TARGET_HEAD].numpy()  # (T, d_head)
k = capture.store["k"][0, TARGET_HEAD].numpy()
v = capture.store["v"][0, TARGET_HEAD].numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5 + 0.15 * T_len))
vmax_qkv = max(abs(q).max(), abs(k).max(), abs(v).max())

for ax, data, title in zip(axes, [q, k, v], ["Q (Query)", "K (Key)", "V (Value)"]):
    im = ax.imshow(data, aspect="auto", cmap="RdBu_r", vmin=-vmax_qkv, vmax=vmax_qkv)
    ax.set_yticks(range(T_len))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(f"d_head = {config.d_head}")

fig.suptitle(f"Q, K, V after projection + RoPE (layer {TARGET_LAYER}, head {TARGET_HEAD})",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Visualize: Attention scores (raw -> masked -> softmax)
scores_raw = capture.store["scores_raw"][0, TARGET_HEAD].numpy()
scores_masked = capture.store["scores_masked"][0, TARGET_HEAD].numpy()
attn_weights = capture.store["attn_weights"][0, TARGET_HEAD].numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Raw scores
vmax_s = abs(scores_raw).max()
im1 = axes[0].imshow(scores_raw, aspect="auto", cmap="RdBu_r", vmin=-vmax_s, vmax=vmax_s)
axes[0].set_title("Raw scores: Q.K^T / sqrt(d)", fontsize=11, fontweight="bold")
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# Masked scores (replace -inf with NaN for display)
display_masked = scores_masked.copy()
display_masked[display_masked == float("-inf")] = float("nan")
cmap_masked = plt.cm.get_cmap("RdBu_r").copy()
cmap_masked.set_bad("#dddddd")
im2 = axes[1].imshow(display_masked, aspect="auto", cmap=cmap_masked, vmin=-vmax_s, vmax=vmax_s)
axes[1].set_title("After causal mask (grey = -inf)", fontsize=11, fontweight="bold")
plt.colorbar(im2, ax=axes[1], shrink=0.8)

# Softmax weights
im3 = axes[2].imshow(attn_weights, aspect="auto", cmap="viridis", vmin=0)
axes[2].set_title("Softmax attention weights\n(each row sums to 1)", fontsize=11, fontweight="bold")
plt.colorbar(im3, ax=axes[2], shrink=0.8)

for ax in axes:
    ax.set_xticks(range(T_len))
    ax.set_yticks(range(T_len))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Key (token j)")
    ax.set_ylabel("Query (token i)")

fig.suptitle(f"Attention mechanism step by step (layer {TARGET_LAYER}, head {TARGET_HEAD})",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Visualize: FFN before/delta/after
ffn_input = capture.store["ffn_input"][0].numpy()   # (T, d_model)
ffn_output = capture.store["ffn_output"][0].numpy()
ffn_delta = ffn_output - ffn_input
show_d_ffn = min(48, config.d_model)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5 + 0.2 * T_len))
vmax_ffn = max(abs(ffn_input[:, :show_d_ffn]).max(), abs(ffn_output[:, :show_d_ffn]).max())
dmax_ffn = abs(ffn_delta[:, :show_d_ffn]).max()

for ax, data, title, vm in zip(
    axes,
    [ffn_input[:, :show_d_ffn], ffn_delta[:, :show_d_ffn], ffn_output[:, :show_d_ffn]],
    ["FFN Input (post-norm)", "Delta (what FFN adds)", "FFN Output"],
    [vmax_ffn, dmax_ffn, vmax_ffn]
):
    im = ax.imshow(data, aspect="auto", cmap="RdBu_r", vmin=-vm, vmax=vm)
    ax.set_yticks(range(T_len))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel(f"First {show_d_ffn} dims")
    plt.colorbar(im, ax=ax, shrink=0.7)

fig.suptitle("Feed-Forward Network: where factual knowledge is stored",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
display(fig)
plt.close(fig)

## Section 6: Training

The training loop uses:
- **AdamW** optimizer with decoupled weight decay (separate decay/no-decay param groups)
- **Linear warmup + cosine decay** learning rate schedule
- **Mixed precision** (bf16 on Ampere+ GPUs, fp16 on older, disabled on CPU)
- **Gradient clipping** to prevent exploding gradients

The implementation is in `train.py` as a **generator** (`train_iter`) that yields structured events. This design lets both the CLI and the Gradio UI share one training implementation.

We train for a small number of epochs here (configurable). On GPU this takes a few minutes; on CPU it will be slower.

In [ ]:
# Visualize the learning rate schedule before training
from train import get_lr

demo_config = NanoLLMConfig()
demo_total_steps = 1000
steps = list(range(demo_total_steps))
lrs = [get_lr(s, demo_config, demo_total_steps) for s in steps]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(steps, lrs, color="#08519c", linewidth=2)
ax.axvline(demo_config.warmup_steps, color="#dc3545", linestyle="--", alpha=0.7,
           label=f"Warmup ends (step {demo_config.warmup_steps})")
ax.set_xlabel("Training step")
ax.set_ylabel("Learning rate")
ax.set_title("Learning Rate Schedule: Linear Warmup + Cosine Decay", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
display(fig)
plt.close(fig)

print(f"Peak LR: {demo_config.learning_rate}")
print(f"Final LR: {lrs[-1]:.2e} (10% of peak -- cosine decay floor)")

In [ ]:
# Training!
# Adjust epochs: 2-3 for quick demo, 15 for full training on GPU.
from train import train_iter

TRAIN_EPOCHS = 3  # ← Increase to 15 for full training on GPU

train_config = NanoLLMConfig()
train_config.max_epochs = TRAIN_EPOCHS
train_config.log_interval = 25

# Collect loss history for plotting
step_losses = []
epoch_summaries = []
generation_samples = {}

print(f"Training for {TRAIN_EPOCHS} epochs on {DEVICE}...")
print(f"(Set TRAIN_EPOCHS = 15 above for full training)\n")

try:
    for evt in train_iter(train_config):
        t = evt["type"]

        if t == "log":
            print(evt["msg"])

        elif t == "step":
            step_losses.append((evt["global_step"], evt["loss"]))
            if evt["batch_idx"] % train_config.log_interval == 0:
                ppl = math.exp(min(evt["loss"], 20))
                print(f"  Epoch {evt['epoch']:>2}/{TRAIN_EPOCHS} | "
                      f"Step {evt['batch_idx']:>4}/{evt['total_batches']} | "
                      f"Loss {evt['loss']:.4f} | PPL {ppl:>8.1f} | "
                      f"LR {evt['lr']:.2e}")

        elif t == "epoch":
            epoch_summaries.append(evt)
            print(f"\n  -- Epoch {evt['epoch']} -- "
                  f"Train: {evt['train_loss']:.4f} | Val: {evt['val_loss']:.4f}")
            if evt["samples"]:
                generation_samples[evt["epoch"]] = evt["samples"]
                print(f"  Sample: \"{evt['samples'][0][:100]}\"")

        elif t == "best":
            print(f"  ★ New best model! val_loss={evt['val_loss']:.4f}")

        elif t == "done":
            print(f"\n{'='*60}")
            print(f"Training complete! Best val loss: {evt['best_val_loss']:.4f}")
            print(f"{'='*60}")

        elif t == "error":
            print(f"ERROR: {evt['msg']}")

except Exception as e:
    print(f"\n⚠ Training interrupted: {e}")
    print(f"  Collected {len(step_losses)} steps before error.")

print(f"\nData collected: {len(step_losses)} steps, {len(epoch_summaries)} epochs")

In [ ]:
# Plot the training loss curve
print(f"Plotting: {len(step_losses)} steps, {len(epoch_summaries)} epochs")

if not step_losses:
    # Fallback: try to load from saved loss curve
    curve_path = os.path.join(config.checkpoint_dir, "loss_curve.png")
    if os.path.exists(curve_path):
        from IPython.display import Image, display
        print(f"No in-memory data, showing saved curve: {curve_path}")
        display(Image(filename=curve_path))
    else:
        print("No training data to plot.")
        print("Run the training cell above first, or check for errors in its output.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Left: per-step loss
    steps_arr = [s for s, _ in step_losses]
    losses_arr = [l for _, l in step_losses]
    axes[0].plot(steps_arr, losses_arr, color="#9ecae1", linewidth=0.7, alpha=0.6, label="Per-step loss")

    if epoch_summaries:
        steps_per_epoch = steps_arr[-1] / max(len(epoch_summaries), 1)
        ep_steps = [e["epoch"] * steps_per_epoch for e in epoch_summaries]
        train_avgs = [e["train_loss"] for e in epoch_summaries]
        val_avgs = [e["val_loss"] for e in epoch_summaries]
        axes[0].plot(ep_steps, train_avgs, "o-", color="#08519c", linewidth=2, markersize=7, label="Train (epoch avg)")
        axes[0].plot(ep_steps, val_avgs, "s-", color="#cb181d", linewidth=2, markersize=7, label="Validation")

    axes[0].set_xlabel("Global step")
    axes[0].set_ylabel("Cross-entropy loss")
    axes[0].set_yscale("log")
    axes[0].set_title("Training Curve", fontsize=13, fontweight="bold")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Right: perplexity per epoch
    if epoch_summaries:
        epochs = [e["epoch"] for e in epoch_summaries]
        train_ppls = [e["train_ppl"] for e in epoch_summaries]
        val_ppls = [e["val_ppl"] for e in epoch_summaries]
        axes[1].plot(epochs, train_ppls, "o-", color="#08519c", linewidth=2, markersize=7, label="Train PPL")
        axes[1].plot(epochs, val_ppls, "s-", color="#cb181d", linewidth=2, markersize=7, label="Val PPL")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Perplexity")
        axes[1].set_title("Perplexity per Epoch", fontsize=13, fontweight="bold")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, "No epoch data yet\n(need ≥1 full epoch)",
                     ha="center", va="center", fontsize=14, transform=axes[1].transAxes)
        axes[1].set_title("Perplexity per Epoch", fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()

## Section 7: Text Generation

Now we use our trained model to generate text! We support two generation modes:

1. **`generate()`** -- standard autoregressive: recomputes the full context at every step. Simple but O(n^2) per token.
2. **`generate_fast()`** -- KV-cached: reuses previously computed K,V vectors. Each step is O(n) instead of O(n^2).

Both use the same sampling strategy:
- **Temperature**: controls randomness (low = deterministic, high = creative)
- **Top-k**: only consider the k most likely tokens
- **Top-p (nucleus)**: only consider tokens whose cumulative probability reaches p

In [ ]:
# Load the best checkpoint (or use the model from training if just trained)
from teach import load_model as teach_load_model

CHECKPOINT_PATH = os.path.join(config.checkpoint_dir, "best.pt")

if os.path.exists(CHECKPOINT_PATH):
    gen_model, gen_tokenizer, gen_config = teach_load_model(CHECKPOINT_PATH, DEVICE)
    print(f"Loaded trained model from {CHECKPOINT_PATH}")
else:
    # Fall back to the model we just trained (or random weights)
    gen_model = model
    gen_tokenizer = tokenizer
    gen_config = config
    gen_model.eval()
    print("No checkpoint found -- using current model (train first for better results).")

In [ ]:
# Generate text with both methods and compare speed
prompts = ["The ", "To be or not to be", "ROMEO:\n", "What is"]

print("=" * 70)
print("Text Generation")
print("=" * 70)

for prompt_text in prompts:
    prompt_tokens = gen_tokenizer.encode(prompt_text, add_special=False)
    prompt_tensor = torch.tensor([prompt_tokens], device=DEVICE)

    # Standard generate (no cache)
    t0 = time.time()
    with torch.no_grad():
        out_std = gen_model.generate(prompt_tensor, max_new_tokens=60,
                                      temperature=0.8, top_k=40, top_p=0.9)
    t_std = time.time() - t0
    text_std = gen_tokenizer.decode(out_std[0].tolist())

    # Fast generate (KV cache)
    t0 = time.time()
    with torch.no_grad():
        out_fast = gen_model.generate_fast(prompt_tensor, max_new_tokens=60,
                                            temperature=0.8, top_k=40, top_p=0.9)
    t_fast = time.time() - t0
    text_fast = gen_tokenizer.decode(out_fast[0].tolist())

    n_new_std = out_std.shape[1] - len(prompt_tokens)
    n_new_fast = out_fast.shape[1] - len(prompt_tokens)

    print(f"\nPrompt: {prompt_text!r}")
    print(f"  generate():      {t_std:.3f}s ({n_new_std / max(t_std, 1e-6):.0f} tok/s)")
    print(f"  generate_fast(): {t_fast:.3f}s ({n_new_fast / max(t_fast, 1e-6):.0f} tok/s)")
    if t_std > 0:
        print(f"  Speedup: {t_std / max(t_fast, 1e-6):.2f}x")
    print(f"  Output: \"{text_fast[:120]}\"")

print("\n" + "=" * 70)

In [ ]:
# Show the effect of sampling parameters
prompt_text = "The little cat"
prompt_tokens = gen_tokenizer.encode(prompt_text, add_special=False)
prompt_tensor = torch.tensor([prompt_tokens], device=DEVICE)

settings = [
    ("Greedy (T=0.01, top_k=1)", 0.01, 1, 1.0),
    ("Conservative (T=0.5, top_k=20)", 0.5, 20, 0.9),
    ("Default (T=0.8, top_k=40)", 0.8, 40, 0.9),
    ("Creative (T=1.2, top_k=100)", 1.2, 100, 0.95),
    ("Wild (T=2.0, top_k=0)", 2.0, 0, 1.0),
]

print(f"Prompt: {prompt_text!r}\n")
torch.manual_seed(42)
for label, temp, topk, topp in settings:
    with torch.no_grad():
        out = gen_model.generate_fast(prompt_tensor.clone(), max_new_tokens=60,
                                       temperature=temp, top_k=topk, top_p=topp)
    text = gen_tokenizer.decode(out[0].tolist())
    print(f"  {label}")
    print(f"    \"{text[:120]}\"")
    print()

## Section 8: Attention Visualization

Let us look at what all 6 layers x 6 heads are attending to. We use `collect_viz_data` from `visualize_anim.py` to run a hooked forward pass across ALL layers and collect attention weights, hidden state norms, and tensor shapes.

Different heads learn different patterns:
- **Previous-token heads**: strongly attend to the immediately preceding token
- **Positional heads**: attend to fixed positions (e.g., first token)
- **Semantic heads**: attend to semantically related tokens regardless of distance

In [ ]:
# Collect visualization data across all layers
from visualize_anim import collect_viz_data

VIZ_PROMPT = "To be or not to be"
viz_data = collect_viz_data(gen_model, gen_tokenizer, VIZ_PROMPT, gen_config)

print(f"Prompt: {VIZ_PROMPT!r}")
print(f"Tokens: {viz_data['tokens']}")
print(f"Layers: {viz_data['n_layers']}, Heads: {viz_data['n_heads']}")
print(f"d_model: {viz_data['d_model']}, d_head: {viz_data['d_head']}")

In [ ]:
# All-heads attention grid: n_layers x n_heads
n_layers = viz_data["n_layers"]
n_heads = viz_data["n_heads"]
tok_labels = viz_data["tokens"]

fig, axes = plt.subplots(n_layers, n_heads, figsize=(3.5 * n_heads, 3 * n_layers))

for layer_idx in range(n_layers):
    for head_idx in range(n_heads):
        ax = axes[layer_idx, head_idx]
        weights = np.array(viz_data["layers"][layer_idx]["attn_weights"][head_idx])
        ax.imshow(weights, cmap="viridis", aspect="auto", vmin=0)
        if layer_idx == 0:
            ax.set_title(f"Head {head_idx}", fontsize=10, fontweight="bold")
        if head_idx == 0:
            ax.set_ylabel(f"Layer {layer_idx}", fontsize=10, fontweight="bold")
        ax.set_xticks(range(len(tok_labels)))
        ax.set_yticks(range(len(tok_labels)))
        if layer_idx == n_layers - 1:
            ax.set_xticklabels(tok_labels, rotation=45, ha="right", fontsize=6)
        else:
            ax.set_xticklabels([])
        if head_idx == 0:
            ax.set_yticklabels(tok_labels, fontsize=6)
        else:
            ax.set_yticklabels([])

fig.suptitle(f"Attention patterns: all {n_layers} layers x {n_heads} heads\n\"{VIZ_PROMPT}\"",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Hidden state norms per layer -- shows how the representation evolves
norms = [viz_data["layers"][i]["hidden_norm"] for i in range(n_layers)]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(n_layers), norms, color="#08519c", edgecolor="white", linewidth=2)
for i, v in enumerate(norms):
    ax.text(i, v + 0.1, f"{v:.1f}", ha="center", fontsize=11, fontweight="bold")
ax.set_xlabel("Layer", fontsize=12)
ax.set_ylabel("Mean L2 norm of hidden state", fontsize=12)
ax.set_title("Activation norms grow through layers\n(residual connections accumulate signal)",
             fontsize=13, fontweight="bold")
ax.set_xticks(range(n_layers))
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
display(fig)
plt.close(fig)

## Section 9: KV Cache Deep Dive

The KV cache is one of the most important optimizations in production LLM inference. Without it, generating N tokens requires N forward passes through the full prompt+generated sequence. With it, each decode step only processes the single newest token.

**How it works:**
1. **Prefill**: Process the entire prompt in one forward pass. Cache all K, V tensors per layer.
2. **Decode**: For each new token, run a single-token forward pass. The cached K, V from all previous steps are concatenated with the new K, V. The query only needs to attend to the new + cached keys.

**The critical invariant**: The RoPE angle for each token must use its **absolute** position, not position 0. When decoding token at position `t`, we pass `start_pos=t` to the RoPE module. Getting this wrong is the #1 way to break the cache.

Let us prove that cached and uncached generation produce identical logits.

In [ ]:
# KV Cache Equivalence Proof
# Full forward on the entire sequence should produce the same logits as
# prefill (T-1 tokens) + 1-token decode with cache.

gen_model.eval()
test_prompt = torch.randint(0, gen_config.vocab_size, (1, 16), device=DEVICE)

with torch.no_grad():
    # Method 1: Full forward pass (no cache)
    full_logits, _ = gen_model(test_prompt)

    # Method 2: Prefill (first T-1 tokens) + decode (last token with cache)
    prefill_logits, cache = gen_model(test_prompt[:, :-1])
    step_logits, _ = gen_model(test_prompt[:, -1:], past_kv=cache)

max_diff = (full_logits - step_logits).abs().max().item()
mean_diff = (full_logits - step_logits).abs().mean().item()

print("KV Cache Equivalence Test")
print("=" * 50)
print(f"  Prompt length:    16 tokens")
print(f"  Full logits shape:  {tuple(full_logits.shape)}")
print(f"  Cache logits shape: {tuple(step_logits.shape)}")
print(f"  Max |difference|:   {max_diff:.2e}")
print(f"  Mean |difference|:  {mean_diff:.2e}")
print(f"  PASS: {'Yes' if max_diff < 1e-4 else 'No'}")

# Show cache structure
print(f"\nCache structure (per layer):")
print(f"  Number of layer caches: {len(cache)}")
k_cache, v_cache = cache[0]
print(f"  K shape: {tuple(k_cache.shape)} -- (batch, n_heads, cached_tokens, d_head)")
print(f"  V shape: {tuple(v_cache.shape)}")

In [ ]:
# Multi-step equivalence: feed tokens one by one through the cache
# and compare against a single full forward pass

MULTI_STEP_LEN = 20
test_seq = torch.randint(0, gen_config.vocab_size, (1, MULTI_STEP_LEN), device=DEVICE)

with torch.no_grad():
    # Reference: full forward pass
    ref_logits, _ = gen_model(test_seq)

    # Step-by-step with cache
    step_logits_list = []
    cache_step = None
    for t in range(MULTI_STEP_LEN):
        token = test_seq[:, t:t+1]
        logits_t, cache_step = gen_model(token, past_kv=cache_step)
        step_logits_list.append(logits_t)

    # The reference logits is only the LAST position; step-by-step gives one logit per step
    # Compare the last-position logit from full pass with last step's logit
    step_last = step_logits_list[-1]
    max_diff_multi = (ref_logits - step_last).abs().max().item()

print(f"Multi-step equivalence ({MULTI_STEP_LEN} tokens fed one at a time):")
print(f"  Max |difference| on final logits: {max_diff_multi:.2e}")
print(f"  PASS: {'Yes' if max_diff_multi < 1e-3 else 'No'}")

In [ ]:
# Timing comparison: cached vs uncached generation at different sequence lengths
prompt_lens = [5, 10, 20, 40]
gen_lens = 50
times_std = []
times_fast = []

for plen in prompt_lens:
    prompt = torch.randint(0, gen_config.vocab_size, (1, plen), device=DEVICE)

    # Standard (uncached)
    t0 = time.time()
    with torch.no_grad():
        _ = gen_model.generate(prompt.clone(), max_new_tokens=gen_lens)
    times_std.append(time.time() - t0)

    # Cached
    t0 = time.time()
    with torch.no_grad():
        _ = gen_model.generate_fast(prompt.clone(), max_new_tokens=gen_lens)
    times_fast.append(time.time() - t0)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(prompt_lens))
width = 0.35
ax.bar(x - width/2, times_std, width, label="generate() -- no cache", color="#dc3545")
ax.bar(x + width/2, times_fast, width, label="generate_fast() -- KV cache", color="#28a745")

for i, (ts, tf) in enumerate(zip(times_std, times_fast)):
    speedup = ts / max(tf, 1e-6)
    ax.text(i, max(ts, tf) + 0.01, f"{speedup:.1f}x", ha="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Prompt length (tokens)")
ax.set_ylabel("Time (seconds)")
ax.set_title(f"KV Cache Speedup (generating {gen_lens} new tokens)", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([str(p) for p in prompt_lens])
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
display(fig)
plt.close(fig)

## Summary

Congratulations! You have built a complete language model from scratch:

| Component | What it does | Where it lives |
|-----------|-------------|----------------|
| **BPE Tokenizer** | Text to/from token IDs | `tokenizer.py` |
| **TextDataset** | Sliding-window (input, target) pairs | `dataset.py` |
| **RMSNorm** | Normalize without centering | `model.py` |
| **RoPE** | Rotary position encoding | `model.py` |
| **CausalSelfAttention** | Multi-head attention with causal mask | `model.py` |
| **FeedForward (SwiGLU)** | Position-wise knowledge storage | `model.py` |
| **TransformerBlock** | Pre-norm residual block | `model.py` |
| **NanoLLM** | Full decoder-only Transformer | `model.py` |
| **Training loop** | AdamW + warmup + cosine decay + AMP | `train.py` |
| **Generation** | Autoregressive sampling with KV cache | `model.py` |

### Next steps
- **Full training**: Set `TRAIN_EPOCHS = 15` and re-run on GPU for much better generation quality
- **Gradio UI**: Run `bash run.sh ui` for an interactive webinar console
- **Teaching slides**: Run `python teach.py` to generate 16 annotated PNG slides
- **Attention visualization**: Run `python visualise.py` for animated attention rollout
- **Explore**: Try different prompts, temperatures, and model sizes!

### References
- Vaswani et al. (2017) *Attention Is All You Need* -- the original Transformer
- Su et al. (2021) *RoFormer: Enhanced Transformer with Rotary Position Embedding* -- RoPE
- Zhang & Sennrich (2019) *Root Mean Square Layer Normalization* -- RMSNorm
- Shazeer (2020) *GLU Variants Improve Transformer* -- SwiGLU
- Press & Wolf (2017) *Using the Output Embedding to Improve Language Models* -- weight tying
- Raschka (2024) *Build a Large Language Model (From Scratch)* -- textbook alignment